# CDS4005 Big Data Analytics
# Individual Assignment (20%), Term 2, 2023-2024


**Student: GUI Shengqi
SID: 4049169**

In [209]:
!pip install pyspark

## Create SparkContext and SparkSession
As always, remeber to create a session, which is an entry point to Spark

In [210]:
from pyspark.sql import SparkSession

ss  = SparkSession.builder \
                            .master("local[1]")\
                            .appName("SparkByExamples.com")\
                            .getOrCreate()
sc = ss.sparkContext

## Question 1: Purchase History Analysis (5%)
Write a module to read the purchase history file (i.e., *car_sales.txt*); then count and display the most frequent **FIVE** car (in their sale count).
The result **MUST** contains the **brand of the top-5 frequent cars and their purchase count”**.
You can return the results in any format (e.g., list, dict), not necessarily in dataframe.

### Sale File Format
The purchase history file will look something like this:

`Ford - - 2015 SUV Auto Black 26`

Each part of this log entry is described below.
* `Ford`: This is the car brand of one sale record. (This is what you need to display!)
* `\-` : The "hyphen" in the output indicates that the requested piece of information (user identity) is not available.
* `\-` : The "hyphen" in the output indicates that the requested piece of information (user DOB) is not available..
* `SUV` : This is the car type of the sale record, we have "SUV", "Passenger", "Hardtop" and so on.
* `Auto`: This means the car is in "Auto mode" or in "Manual mode".
* `Black` : This is the color of the car.
* `26` : This is the sales price of the car (26 means 26,000$).


### Your Answer for Q1:

In [211]:
# these line are used to load the Google drive into the notebook
from google.colab import drive
drive.mount('/content/drive/')
data_path = "/content/drive/MyDrive/Colab Notebooks/Big Data/Assignment 1/"  # this is your drive

textFile = sc.textFile(data_path + "car_sales.txt")
textFile.collect()

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


['Ford - - 2015 SUV Auto Black 26',
 'Dodge - - 2015 SUV Auto Black 19',
 'Cadillac - - 2015 Passenger Manual Red 31',
 'Toyota - - 2016 SUV Manual "Pale White" 14',
 'Acura - - 2015 Hatchback Auto Red 24',
 'Mitsubishi - - 2016 Hatchback Manual Red 12',
 'Toyota - - 2016 Passenger Manual Red 14',
 'Volvo - - 2015 Sedan Auto "Pale White" 42',
 'Mercury - - 2016 Hardtop Auto "Pale White" 21',
 'Buick - - 2015 Hatchback Auto Black 61',
 'Mitsubishi - - 2015 Hardtop Auto Red 39',
 'Saturn - - 2016 Passenger Auto Red 25',
 'Volvo - - 2015 Hatchback Manual Black 17',
 'Mercury - - 2016 Hardtop Manual Red 21',
 'Cadillac - - 2015 SUV Auto Red 22',
 'Jaguar - - 2015 Passenger Manual Red 45',
 'Mitsubishi - - 2016 Hardtop Manual Black 25',
 'Ford - - 2016 Sedan Auto "Pale White" 62',
 'Volkswagen - - 2015 Passenger Auto Black 22',
 'Mitsubishi - - 2015 Hardtop Auto Black 45',
 'Lexus - - 2016 Hardtop Auto Black 16',
 'BMW - - 2016 Sedan Auto Black 16',
 'Subaru - - 2016 SUV Auto Red 57',
 'Bui

In [212]:
# Split by each line
each_line = textFile.flatMap(lambda x: x.split('/n'))
each_line.collect()

['Ford - - 2015 SUV Auto Black 26',
 'Dodge - - 2015 SUV Auto Black 19',
 'Cadillac - - 2015 Passenger Manual Red 31',
 'Toyota - - 2016 SUV Manual "Pale White" 14',
 'Acura - - 2015 Hatchback Auto Red 24',
 'Mitsubishi - - 2016 Hatchback Manual Red 12',
 'Toyota - - 2016 Passenger Manual Red 14',
 'Volvo - - 2015 Sedan Auto "Pale White" 42',
 'Mercury - - 2016 Hardtop Auto "Pale White" 21',
 'Buick - - 2015 Hatchback Auto Black 61',
 'Mitsubishi - - 2015 Hardtop Auto Red 39',
 'Saturn - - 2016 Passenger Auto Red 25',
 'Volvo - - 2015 Hatchback Manual Black 17',
 'Mercury - - 2016 Hardtop Manual Red 21',
 'Cadillac - - 2015 SUV Auto Red 22',
 'Jaguar - - 2015 Passenger Manual Red 45',
 'Mitsubishi - - 2016 Hardtop Manual Black 25',
 'Ford - - 2016 Sedan Auto "Pale White" 62',
 'Volkswagen - - 2015 Passenger Auto Black 22',
 'Mitsubishi - - 2015 Hardtop Auto Black 45',
 'Lexus - - 2016 Hardtop Auto Black 16',
 'BMW - - 2016 Sedan Auto Black 16',
 'Subaru - - 2016 SUV Auto Red 57',
 'Bui

In [213]:
# Split brand and information by ' - - '
splited_lines = each_line.map(lambda x: x.split(" - - "))
lines_df = ss.createDataFrame(splited_lines, ['brand', 'information'])
lines_df.show()

+----------+--------------------+
|     brand|         information|
+----------+--------------------+
|      Ford|2015 SUV Auto Bla...|
|     Dodge|2015 SUV Auto Bla...|
|  Cadillac|2015 Passenger Ma...|
|    Toyota|2016 SUV Manual "...|
|     Acura|2015 Hatchback Au...|
|Mitsubishi|2016 Hatchback Ma...|
|    Toyota|2016 Passenger Ma...|
|     Volvo|2015 Sedan Auto "...|
|   Mercury|2016 Hardtop Auto...|
|     Buick|2015 Hatchback Au...|
|Mitsubishi|2015 Hardtop Auto...|
|    Saturn|2016 Passenger Au...|
|     Volvo|2015 Hatchback Ma...|
|   Mercury|2016 Hardtop Manu...|
|  Cadillac|2015 SUV Auto Red 22|
|    Jaguar|2015 Passenger Ma...|
|Mitsubishi|2016 Hardtop Manu...|
|      Ford|2016 Sedan Auto "...|
|Volkswagen|2015 Passenger Au...|
|Mitsubishi|2015 Hardtop Auto...|
+----------+--------------------+
only showing top 20 rows



In [214]:
# Change the py_dict to rdd
count_dict = lines_df.rdd.countByKey()
count_dict

defaultdict(int,
            {'Ford': 167,
             'Dodge': 173,
             'Cadillac': 64,
             'Toyota': 114,
             'Acura': 80,
             'Mitsubishi': 146,
             'Volvo': 79,
             'Mercury': 92,
             'Buick': 42,
             'Saturn': 59,
             'Jaguar': 16,
             'Volkswagen': 131,
             'Lexus': 85,
             'BMW': 87,
             'Subaru': 44,
             'Chevrolet': 177,
             'Nissan': 88,
             'Plymouth': 59,
             'Hyundai': 32,
             'Porsche': 34,
             'Oldsmobile': 113,
             'Honda': 72,
             'Pontiac': 79,
             'Lincoln': 49,
             'Mercedes-B': 126,
             'Infiniti': 19,
             'Jeep': 33,
             'Chrysler': 111,
             'Saab': 21,
             'Audi': 51})

In [215]:
# Count each brand
count_frame = ss.createDataFrame(count_dict.items(), ['brand', 'counts'])
count_frame.show()

+----------+------+
|     brand|counts|
+----------+------+
|      Ford|   167|
|     Dodge|   173|
|  Cadillac|    64|
|    Toyota|   114|
|     Acura|    80|
|Mitsubishi|   146|
|     Volvo|    79|
|   Mercury|    92|
|     Buick|    42|
|    Saturn|    59|
|    Jaguar|    16|
|Volkswagen|   131|
|     Lexus|    85|
|       BMW|    87|
|    Subaru|    44|
| Chevrolet|   177|
|    Nissan|    88|
|  Plymouth|    59|
|   Hyundai|    32|
|   Porsche|    34|
+----------+------+
only showing top 20 rows



In [216]:
# Sort the result
count_frame.sort('counts')
count_frame.sort('counts', ascending=False).show(5)

+----------+------+
|     brand|counts|
+----------+------+
| Chevrolet|   177|
|     Dodge|   173|
|      Ford|   167|
|Mitsubishi|   146|
|Volkswagen|   131|
+----------+------+
only showing top 5 rows



## Question 2: Text Classify (15%)
In this task, you are given a **JSON file** that store research paper data. Your task is to use any Pyspark and/or Python operators to write a module that:

Using only the **Section Header Name**, classify each paper section into one of the following type: introduction (**i**), method (**m**), result (**r**), conclusion (**c**), discussion (**d**). Given the Keyword List, the classification is done using **Keyword Exact Match**.


**Question 2.a Begainning**

In [217]:
import json
from pyspark.sql.types import StructType,StructField, StringType, IntegerType,BooleanType,DoubleType

KEYWORDS = {
    'introduction': 'i',
    'case': 'i',
    'purpose': 'i',
    'objective': 'i',
    'objectives': 'i',
    'aim': 'i',
    'summary': 'i',
    'findings': 'i',
    'background': 'i',
    'background/aims': 'i',
    'literature': 'i',
    'studies': 'i',
    'related': 'i',
    'methods': 'm',
    'method': 'm',
    'techniques': 'm',
    'methodology': 'm',
    'results': 'r',
    'result': 'r',
    'experiment': 'r',
    'experiments': 'r',
    'experimental': 'r',
    'discussion': 'd',
    'limitations': 'd',
    'conclusion': 'c',
    'conclusions': 'c',
    'concluding': 'c'
}



In [218]:
multiline_dataframe = ss.read.option("multiline","true") \
                      .json(data_path + "assignment.text.20.json")
multiline_dataframe.show(5)

+--------------------+----------+--------------------+--------------------+--------------------+
|       abstract_text|article_id|        article_text|       section_names|            sections|
+--------------------+----------+--------------------+--------------------+--------------------+
|[<S> the short - ...|         0|[for about 20 yea...|[introduction, me...|[[for about 20 ye...|
|[<S> we study the...|         1|[it is believed t...|[introduction, st...|[[it is believed ...|
|[<S> starting fro...|         2|[as a common quan...|[[sec:intro]intro...|[[as a common qua...|
|[<S> we study a n...|         3|[for the hybrid m...|[introduction, ge...|[[for the hybrid ...|
|[<S> new methods ...|         4|[recently it was ...|[introduction, de...|[[recently it was...|
+--------------------+----------+--------------------+--------------------+--------------------+
only showing top 5 rows



In [219]:
multiline_dataframe.printSchema()

root
 |-- abstract_text: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- article_id: long (nullable = true)
 |-- article_text: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- section_names: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- sections: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: string (containsNull = true)



In [220]:
from pyspark.sql.functions import explode, concat_ws, col, lower

# Split the DataFrame by 'section_names'
df1 = multiline_dataframe.select(
    "abstract_text",
    "article_id",
    "article_text",
    explode("section_names").alias("section_head")
)

# Split the DataFrame by 'sections'
df2 = multiline_dataframe.select(
    "abstract_text",
    "article_id",
    "article_text",
    "section_names",
    explode("sections").alias("section_sentences_list")
)

# Stringify and lowercased the section_document content
df2_transformed = df2.select(concat_ws(" ", col("section_sentences_list")).alias("paragraph").cast("string"))
df2_lower = df2_transformed.select(lower(col("paragraph")).alias("section_document"))

#Show the processing result
df1.show(5)
df2_lower.show(5)
print(df1.count())
print(df2_lower.count())

+--------------------+----------+--------------------+--------------------+
|       abstract_text|article_id|        article_text|        section_head|
+--------------------+----------+--------------------+--------------------+
|[<S> the short - ...|         0|[for about 20 yea...|        introduction|
|[<S> the short - ...|         0|[for about 20 yea...|methods of period...|
|[<S> the short - ...|         0|[for about 20 yea...|a method of the d...|
|[<S> the short - ...|         0|[for about 20 yea...|       data analysis|
|[<S> the short - ...|         0|[for about 20 yea...|the periodicity a...|
+--------------------+----------+--------------------+--------------------+
only showing top 5 rows

+--------------------+
|    section_document|
+--------------------+
|for about 20 year...|
|to find periodici...|
|the method of the...|
|in this paper the...|
|in order to verif...|
+--------------------+
only showing top 5 rows

120
120


In [221]:
import pandas as pd
# Use pandas to merge the 2 DataFrames

pandas_df1 = df1.toPandas()
pandas_df2_lower = df2_lower.toPandas()

combined_df = pd.concat([pandas_df1, pandas_df2_lower], axis=1)

flated_df = ss.createDataFrame(combined_df)
flated_df.show(5)

+--------------------+----------+--------------------+--------------------+--------------------+
|       abstract_text|article_id|        article_text|        section_head|    section_document|
+--------------------+----------+--------------------+--------------------+--------------------+
|[<S> the short - ...|         0|[for about 20 yea...|        introduction|for about 20 year...|
|[<S> the short - ...|         0|[for about 20 yea...|methods of period...|to find periodici...|
|[<S> the short - ...|         0|[for about 20 yea...|a method of the d...|the method of the...|
|[<S> the short - ...|         0|[for about 20 yea...|       data analysis|in this paper the...|
|[<S> the short - ...|         0|[for about 20 yea...|the periodicity a...|in order to verif...|
+--------------------+----------+--------------------+--------------------+--------------------+
only showing top 5 rows



In [222]:
# Define a UDF that classifies each section name in a list
def classify_sections(section_names):
  section_splited = section_names.split()
  section_class = 'Others'
  for section_name in section_splited:
    if section_name.lower() in KEYWORDS:
      section_class = KEYWORDS[section_name.lower()]
    else:
      continue
  return section_class

**Question 2.a Result**

In [223]:
from pyspark.sql.functions import udf

# Definit a UDF
classify_sections_udf = udf(classify_sections, StringType())

# Apply the UDF to classify each section name in the lists
classified_sections_df = flated_df.withColumn("section_id", classify_sections_udf(flated_df['section_head']))

# Show some results to verify
classified_sections_df.select("section_head", "section_id").show(truncate=False)

+------------------------------------------------------------------------------+----------+
|section_head                                                                  |section_id|
+------------------------------------------------------------------------------+----------+
|introduction                                                                  |i         |
|methods of periodicity analysis                                               |m         |
|a method of the diagnosis of an echo-effect in the power spectrum             |m         |
|data analysis                                                                 |Others    |
|the periodicity at about 155 days during the maximum activity period          |Others    |
|conclusion                                                                    |c         |
|acknowledgments                                                               |Others    |
|introduction                                                                  |

**Question 2.b**

In [224]:
# Filter out 'Others' and show the result
solution_b = classified_sections_df.filter(classified_sections_df.section_id != 'Others').select('article_id',
                                                                                                 'section_id',
                                                                                                 'section_head',
                                                                                                 'section_document')
solution_b.show()

+----------+----------+--------------------+--------------------+
|article_id|section_id|        section_head|    section_document|
+----------+----------+--------------------+--------------------+
|         0|         i|        introduction|for about 20 year...|
|         0|         m|methods of period...|to find periodici...|
|         0|         m|a method of the d...|the method of the...|
|         0|         c|          conclusion|the following res...|
|         1|         i|        introduction|it is believed th...|
|         1|         c|          conclusion|we have studied t...|
|         2|         c|          conclusion|in the study of b...|
|         3|         i|        introduction|for the hybrid mo...|
|         3|         r|   numerical results|in this section w...|
|         3|         c|conclusions and o...|we presented the ...|
|         4|         i|        introduction|recently it was d...|
|         4|         c|         conclusions|finally , we summ...|
|         

**Question 2.c**

In [225]:
from pyspark.sql.functions import col, size, split

# Filter out "Others"
filtered_df = solution_b.filter(solution_b.section_id != "Others")

# Compute the number of words in section_document
# Assume that the words splited by space
words_df = filtered_df.withColumn("word_count", size(split(col("section_document"), " ")))

# Fiter out the words number more than 50
final_df = words_df.filter(words_df.word_count > 50)

# Ascending by the 'article_id'
sorted_df = final_df.orderBy("article_id").drop('word_count')

# Show result
sorted_df.show(truncate=False)

+----------+----------+-----------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

**Question 2.d**

In [226]:
# Save the DataFrame to JSON file
path_to_save_json = data_path + "my_output_dataframe.json"
sorted_df.write.json(path_to_save_json, mode="overwrite")